# Federated Deep Learning for Financial Fraud Detection using TensorFlow

This notebook demonstrates how to implement federated learning for financial fraud detection using NVIDIA FLARE (NVFlare). The notebook shows a complete workflow for training a deep learning model across multiple clients while maintaining data privacy.

## 0. Define User Info
Update the user information based on your details and the MLFlow server provided.

In [ ]:
#startup_kit_location = "/path/to/admin_startup_kit"
#username = "admin@nvidia.com"

# MLFlow tracking info
mlflow_tracking_uri = None #  if None, no metrics will be streamed

## 1. Define Recipe

By default, we can use a small randomly generated dataset to illustrate the workflow. If the full sythentic dataset should be used, please provide the filepath with the `--dataset` train argument.

In [ ]:
import os
from train.model import SimpleNetwork

from nvflare.app_opt.pt.recipes.fedavg import FedAvgRecipe
from nvflare.app_opt.pt.recipes.fedopt import FedOptRecipe
from nvflare.recipe import SimEnv, ProdEnv
from misc.data import all_model_parameters


result_dir = "results"

# Hyperparams
num_rounds = 20
min_clients = 5  # should be all clients even if some cause an error
local_epochs = 1
lr = 5e-4
batch_size = 64
width_factor = 1
dropout_p = 0.2
algo = "fedavg"  # "fedavg", "fedprox", "fedopt", or "local"
fedproxloss_mu = 0 #1e-3

data_selection="sim-exp" 
# data paths are defined in misc/experiments.py

# Differential Privacy Configuration
enable_dp = False  # Set to True to enable differential privacy
# Privacy budget (lower = more privacy, less accuracy)
# typical epsilon values: strong privacy: 1.0, moderate privacy: 10.0, weak privacy: 50.0)
target_epsilon = 50.0
target_delta = 1e-5    # Privacy parameter. "Generally, it should be set to be less than the inverse of the size of the training dataset" (from https://opacus.ai/tutorials/building_image_classifier)
max_grad_norm = 1.0   # Gradient clipping threshold for DP

Define the recipe.

In [ ]:
train_args = f"--data_selection={data_selection} --epochs={local_epochs} --lr {lr} --batch-size {batch_size}"

name = f"{algo}_rounds{num_rounds}_lr{lr}_lepochs{local_epochs}"
if algo in ["fedavg", "fedopt"]:
    pass
elif algo == "fedprox":
    if fedproxloss_mu > 0:
        name += f"{fedproxloss_mu}"
        train_args += f" --fedproxloss_mu {fedproxloss_mu}"
    else:
        raise ValueError("Select a fedproxloss_mu > 0 when using FedProx")
elif algo == "local":
    train_args += f" --local_only={True}"
else:
    raise ValueError(f"Algo `{algo}` not supported!")

# Add differential privacy parameters if enabled
if enable_dp:
    dp_params = (
        f" --enable_dp={True}"
        f" --target_epsilon={target_epsilon}"
        f" --target_delta={target_delta}"
        f" --max_grad_norm={max_grad_norm}"
    )
    train_args += dp_params
    name += f"_dp{target_epsilon}"
    print(f"Differential Privacy enabled:")
    print(f"  - ε (epsilon): {target_epsilon}")
    print(f"  - δ (delta): {target_delta}")
    print(f"  - max_grad_norm: {max_grad_norm}")
else:
    print("Differential Privacy disabled")

name = f"fsi-{name}-{data_selection}"

if algo in ["local", "fedavg", "fedprox"]:
    from nvflare.apis.dxo import DataKind
    print (f"Using FedAvgRecipe for Algo: {algo}")
    # Define recipe
    n_features = len(all_model_parameters)
    print(f"Using n_features: {n_features}")
    recipe = FedAvgRecipe(
        name=name,
        min_clients=min_clients,  # should be all clients even if some cause an error
        num_rounds=num_rounds,
        model=SimpleNetwork(input_size=n_features, num_classes=2, width_factor=width_factor, dropout_p=dropout_p),
        train_script=os.path.join(os.getcwd(),"..", "train", "client.py"),
        train_args=train_args,
        aggregator_data_kind=DataKind.WEIGHT_DIFF
    )
elif algo == "fedopt":
    print (f"Using FedOptRecipe for Algo: {algo}")
    # Define recipe
    n_features = len(all_model_parameters)
    print(f"Using n_features: {n_features}")
    recipe = FedOptRecipe(
        name=name,
        min_clients=min_clients,  # should be all clients even if some cause an error
        num_rounds=num_rounds,
        model=SimpleNetwork(input_size=n_features, num_classes=2, width_factor=width_factor, dropout_p=dropout_p),
        train_script=os.path.join(os.getcwd(),"..", "train", "client.py"),
        train_args=train_args,
        optimizer_args={
            "path": "torch.optim.SGD",
            "args": {"lr": 0.9, "momentum": 0.2},
            "config_type": "dict"
        },
        lr_scheduler_args={
            "path": "torch.optim.lr_scheduler.CosineAnnealingLR",
            "args": {"T_max": "{num_rounds}", "eta_min": 0.5},
            "config_type": "dict"
        }
    )

### Add Experiment Tracking
We can add a central MLFlow tracker where clients can report their training progress.

In [ ]:
from nvflare.recipe.utils import add_experiment_tracking

if mlflow_tracking_uri:
    add_experiment_tracking(recipe, tracking_type="mlflow", tracking_config={"tracking_uri": mlflow_tracking_uri, "kw_args": {"experiment_name": "FLARE Simulation"}})

### Add filter to collect SHAP metrics on the server
We add a filter to the server to collect the incoming SHAP metrics reported by the clients.

In [ ]:
from train.utils import MetricsCollectionFilter
shap_filter = MetricsCollectionFilter()
recipe.add_server_input_filter(shap_filter, tasks=["train"])

# optionally export
recipe.export("job_configs")

## 2. Run in Simulation Environment

In [ ]:
site_names = ["siteA", "siteB", "siteC", "siteD", "siteE"]
env = SimEnv(clients=site_names, num_threads=len(site_names))
run = recipe.execute(env=env)

## 3. (Optionally) Run in Production Environment

In [ ]:
#env = ProdEnv(startup_kit_location=startup_kit_location, username=username)
#run = recipe.execute(env=env)

### Get Status

In [ ]:
print("Job Status is:", run.get_status())

### Get Results

In [ ]:
result_path = run.get_result()
print("Result can be found in:", result_path)

### Optionally Abort Job
If you'd like to stop the job before completion for any reason, you can run:

In [ ]:
#run.abort()

### 4. Visualize SHAP metrics
In this example, the server collects the clients' SHAP metrics at each round and saves them to a file. After downloading the training artifacts from the server (using `run.get_result()`), we can access them here and plot the results.

In [ ]:
# ProdEnv
# metrics_file = result_path + "/workspace/app_server/metrics.npy"
# SimEnv
metrics_file = result_path + "/server/simulate_job/app_server/metrics.npy"
client = "siteA"
num_round = num_rounds - 1

import matplotlib.pyplot as plt
%matplotlib inline

from train.utils import plot_attribution_summary, plot_attribution_feature_importance, load_numpy

all_metrics = load_numpy(metrics_file)
print(f"All metrics: {all_metrics.keys()}")

shap_metrics = all_metrics[f"round{num_round}"][client]["shap_metrics"]
print(f"SHAP metrics for round {num_round} and site-1: {shap_metrics.keys()}")   

#### Show summary plot

In [ ]:
plot_attribution_summary(shap_metrics)
plt.title(f"SHAP Values Round {num_round} for Client {client}")
plt.show()

#### Show feature importance plot

In [ ]:
plot_attribution_feature_importance(shap_metrics)
plt.title(f"Feature Importance Round {num_round} for Client {client}")
plt.show()